# Solution B1 · XGBoost with early stopping on house prices

**Type**: Implement with scikit-learn

## Objective

Train an `XGBRegressor` on California with **early stopping** on a validation set, compare three learning rates (0.3, 0.1, 0.03) for chosen rounds, RMSE and time, and place the result against linear regression, tree and random forest (previous modules). Check that halving the learning rate requires more rounds.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
print(X_tr.shape, X_val.shape, X_te.shape)

## Commented solution

In [ ]:
rows, models = {}, {}
for eta in [0.3, 0.1, 0.03]:
    m = xgb.XGBRegressor(n_estimators=4000, learning_rate=eta, max_depth=5, subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                         early_stopping_rounds=50, eval_metric="rmse", n_jobs=-1, random_state=0)
    t0 = time.time()
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    rows[eta] = {"best_round": m.best_iteration + 1, "val_rmse": m.best_score, "test_rmse": mean_squared_error(y_te, m.predict(X_te), squared=False), "time_s": time.time() - t0}
    models[eta] = m
table = pd.DataFrame(rows).T
table.round(4)

In [ ]:
eta_round_product = pd.Series({eta: eta * table.loc[eta, "best_round"] for eta in table.index})
round_ratio_003_over_03 = table.loc[0.03, "best_round"] / table.loc[0.3, "best_round"]
print(eta_round_product.round(1).to_string(), f"\nratio of the rounds between eta 0.03 and 0.3: {round_ratio_003_over_03:.1f}")

In [ ]:
best = table["val_rmse"].idxmin()
error = lambda m: mean_squared_error(y_te, m.predict(X_te), squared=False)
reference_rmse = {"linear": error(LinearRegression().fit(X_tr, y_tr)), "tree": error(DecisionTreeRegressor(min_samples_leaf=20, random_state=0).fit(X_tr, y_tr)),
                  "forest": error(RandomForestRegressor(200, max_features=0.33, random_state=0, n_jobs=-1).fit(X_tr, y_tr)), "xgb": table.loc[best, "test_rmse"]}
val_test_gap = float(table.loc[best, "test_rmse"] - table.loc[best, "val_rmse"])
print({k: round(v, 3) for k, v in reference_rmse.items()}, f"| test - validation gap: {val_test_gap:+.4f} (best learning rate: {best})")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["best_round", "val_rmse", "test_rmse", "time_s"] and list(table.index) == [0.3, 0.1, 0.03], "table with three learning rates and four columns", "best_round, val_rmse, test_rmse, time_s")
    check(table["best_round"].is_monotonic_increasing and table.loc[0.03, "best_round"] > 5 * table.loc[0.3, "best_round"], "lower learning rate = more rounds: at 0.03 over 5 times those at 0.3", "best_iteration + 1")
    check(table["test_rmse"].max() < 0.5 and table["test_rmse"].min() > 0.4 and table.loc[0.03, "test_rmse"] <= table.loc[0.3, "test_rmse"] + 0.005, "test RMSE between 0.40 and 0.50; the low learning rate does not get worse", "XGBRegressor with early stopping")
    check(close(eta_round_product.to_numpy(), table.index.to_numpy() * table["best_round"].to_numpy()) and eta_round_product.max() / eta_round_product.min() < 6 and round_ratio_003_over_03 > 5, "the eta x round product varies by less than 6 times across the learning rates (the rounds scale almost like 1/eta); round ratio over 5", "eta * best_round")
    check(reference_rmse["xgb"] < reference_rmse["forest"] < reference_rmse["tree"] < reference_rmse["linear"], "order: xgb < forest < tree < linear", "compare the test RMSEs")
    check(0 <= val_test_gap < 0.03, "the gap between test and validation RMSE is small (less than 0.03)", "the validation set only decided the number of rounds")
done()

## What you learned

- **The standard workflow:** low learning rate, high `n_estimators` as a ceiling, `early_stopping_rounds`, best round from `best_iteration`, test set only at the end. You do not need to guess the number of rounds: 4000 is a ceiling that early stopping never touches (the real rounds are 149, 843 and 1985 for eta 0.3, 0.1 and 0.03).
- **`learning_rate` and rounds are linked:** with eta 0.3 tens of rounds are enough, with 0.03 you need hundreds or thousands (about 1/eta). The `eta × round` product stays of the same order of magnitude: the "total amount of correction". With a small eta the generalization is better (slightly) and the time grows.
- **The ranking of the models is the expected one:** linear (0.734) > optimized tree (0.613) > forest (0.505) > XGBoost (0.457). XGBoost reduces the error by about a tenth compared with the forest: that is why it is the reference model on tabular data.
- **The validation-test gap is small** (0.009 of RMSE): the validation set only decided *when to stop* (one parameter), not the weights of the tree. If you had used the test set as validation the estimate would have been optimistic by this same small amount: a good reason not to, and to put the validation set **last** in `eval_set` (exercise E1).

## Common pitfall

Using `n_estimators` as a hyperparameter to search on a grid together with `learning_rate`: the two things compensate each other. Fix the learning rate, keep `n_estimators` high and let early stopping decide.

## Interview question

<details><summary>How do you choose the number of trees in XGBoost?</summary>

I fix a low learning rate (0.03-0.1), allow many trees (thousands) and use early stopping on a separate validation set: the training stops when the metric does not improve for k rounds and the model uses the best one. The number of rounds is therefore a result, not a hyperparameter to search; for the final estimate I use a test set never seen,
or a cross-validation with early stopping inside each fold.

</details>